# HOXC6_KO vs WT Unique SNP Extractor
Google Drive의 대용량 SNP annotation 파일을 읽어 WT에 없는 HOXC6_KO SNP를 exact variant key 기준으로 추출합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os,csv,gzip,shutil,subprocess,time
from pathlib import Path

ROOT=Path('/content/drive/MyDrive/HOXC6_KO_variant_analysis')
KO_FILE=ROOT/'HOXC6_KO.GATK.snp.annovar.hg38_multianno.xls'
WT_FILE=ROOT/'WT.GATK.snp.annovar.hg38_multianno.xls'
OUT_DIR=ROOT/'SNP_unique_vs_WT'; OUT_DIR.mkdir(parents=True,exist_ok=True)
TMP_DIR=Path('/content/snp_unique_tmp'); TMP_DIR.mkdir(parents=True,exist_ok=True)
KO_KEYS=TMP_DIR/'ko.keys.tsv'; WT_KEYS=TMP_DIR/'wt.keys.tsv'
KO_SORT=TMP_DIR/'ko.keys.sorted.tsv'; WT_SORT=TMP_DIR/'wt.keys.sorted.tsv'
UNIQUE_KEYS=TMP_DIR/'ko_unique.keys.tsv'
UNIQUE_OUT=OUT_DIR/'HOXC6_KO_unique_SNP_vs_WT.hg38_multianno.tsv'
GZ_OUT=Path(str(UNIQUE_OUT)+'.gz')
ZIP_OUT=ROOT/'HOXC6_KO_unique_SNP_vs_WT_hg38.zip'
for p in [KO_FILE,WT_FILE]:
    if not p.exists(): raise FileNotFoundError(p)
    print(p.name, f'{p.stat().st_size/1024**3:.2f} GB')


In [ ]:
def get_header(path):
    with open(path,'r',encoding='utf-8',errors='replace') as f:
        return f.readline().rstrip('\n').split('\t')
cands=[['Chr','Start','End','Ref','Alt'],['CHROM','POS','REF','ALT'],['Chr','Start','Ref','Alt']]
def detect(h):
    for c in cands:
        if all(x in h for x in c): return c
    raise ValueError(f'Cannot detect key columns: {h[:30]}')
KO_COLS=detect(get_header(KO_FILE)); WT_COLS=detect(get_header(WT_FILE))
print('KO key',KO_COLS); print('WT key',WT_COLS)
if KO_COLS!=WT_COLS: raise ValueError('KO/WT key columns differ')


In [ ]:
def write_keys(src,dst,cols):
    n=0; t=time.time()
    with open(src,'r',encoding='utf-8',errors='replace',newline='') as fi, open(dst,'w',encoding='utf-8') as fo:
        r=csv.reader(fi,delimiter='\t'); h=next(r); idx=[h.index(c) for c in cols]
        for row in r:
            if row and len(row)>max(idx):
                fo.write('\t'.join(row[i] for i in idx)+'\n'); n+=1
                if n%1000000==0: print(src.name,f'{n:,}',f'{(time.time()-t)/60:.1f} min')
    print(src.name,'total',f'{n:,}'); return n
ko_rows=write_keys(KO_FILE,KO_KEYS,KO_COLS)
wt_rows=write_keys(WT_FILE,WT_KEYS,WT_COLS)


In [ ]:
env=os.environ.copy(); env['LC_ALL']='C'
for src,dst in [(KO_KEYS,KO_SORT),(WT_KEYS,WT_SORT)]:
    subprocess.run(['sort','-T',str(TMP_DIR),'-S','60%','--parallel=2','-u',str(src),'-o',str(dst)],check=True,env=env)
with open(UNIQUE_KEYS,'w') as fo:
    subprocess.run(['comm','-23',str(KO_SORT),str(WT_SORT)],stdout=fo,check=True,env=env)
with open(UNIQUE_KEYS) as f: unique_n=sum(1 for _ in f)
print('KO-only unique SNP keys:',f'{unique_n:,}')


In [ ]:
unique=set()
with open(UNIQUE_KEYS,encoding='utf-8') as f:
    for line in f: unique.add(line.rstrip('\n'))
with open(KO_FILE,'r',encoding='utf-8',errors='replace',newline='') as fi, open(UNIQUE_OUT,'w',encoding='utf-8',newline='') as fo:
    r=csv.reader(fi,delimiter='\t'); w=csv.writer(fo,delimiter='\t',lineterminator='\n')
    h=next(r); w.writerow(h); idx=[h.index(c) for c in KO_COLS]
    scanned=written=0
    for row in r:
        scanned+=1
        if row and len(row)>max(idx):
            key='\t'.join(row[i] for i in idx)
            if key in unique: w.writerow(row); written+=1
        if scanned%1000000==0: print('scanned',f'{scanned:,}','written',f'{written:,}')
print('output rows',f'{written:,}')


In [ ]:
with open(UNIQUE_OUT,'rb') as fi, gzip.open(GZ_OUT,'wb',compresslevel=6) as fo:
    shutil.copyfileobj(fi,fo,length=8*1024*1024)
(OUT_DIR/'summary.txt').write_text(f'''HOXC6_KO vs WT Unique SNP Extraction\nGenome: hg38\nVariant matching key: {', '.join(KO_COLS)}\nKO rows: {ko_rows:,}\nWT rows: {wt_rows:,}\nKO-only unique SNP keys: {unique_n:,}\nOutput annotation rows: {written:,}\n''',encoding='utf-8')
(OUT_DIR/'README.txt').write_text(f'''Exact KO-vs-WT SNP comparison.\nKey: {', '.join(KO_COLS)}\nOriginal KO annotation columns are preserved.\n''',encoding='utf-8')
if ZIP_OUT.exists(): ZIP_OUT.unlink()
shutil.make_archive(str(ZIP_OUT.with_suffix('')),'zip',root_dir=OUT_DIR)
print('DONE',ZIP_OUT)
print('ZIP MB',f'{ZIP_OUT.stat().st_size/1024**2:.2f}')
